# COGS 188 — Project Proposal
## University Course Timetabling with Search and Evolutionary Optimization

**Group members:** Derrick Chen, Zhen Wang, and Daira Merino  
**Project type:** Implementation  
**Repository:** [zhc066/COGS188_group](https://github.com/zhc066/COGS188_group)

# Abstract

We propose to compare AI search methods for constructing university course timetables. The task is to assign weekly course meetings to rooms and periods while avoiding room, instructor, and student-cohort conflicts. Among feasible schedules, we will minimize penalties for inconvenient gaps, undesirable meeting times, and changes of room within a course. We will generate 1,200 reproducible synthetic instances with 20–100 courses and three levels of resource contention, measured as the fraction of room-periods that must be filled. A greedy scheduler will provide a baseline; starting from its timetable, we will compare simulated annealing, a genetic algorithm, and a hill-climbing ablation to measure how much each improves on greedy. We will evaluate feasibility rate, preference penalties, runtime, and consistency across random seeds under equal computation budgets. Our central question is which method most reliably produces feasible, low-penalty timetables as resource contention increases. The intended outcome is a reproducible comparison and timetable visualization, rather than a production scheduling system.

# Background

Every term, universities must place hundreds of course meetings into a limited set of rooms and time slots, and the resulting timetable shapes students' days, instructors' workloads, and how efficiently rooms are used. Automated timetabling has been studied for decades in operations research and AI; one survey distinguishes school, course, and examination timetabling and catalogs the hard and soft constraints each involves<a name="schaerf"></a>[<sup>[1]</sup>](#schaerfnote), and a later review describes newer approaches to university course and exam timetabling<a name="burke"></a>[<sup>[2]</sup>](#burkenote). Even the feasibility question for simplified timetabling problems is NP-complete<a name="even"></a>[<sup>[3]</sup>](#evennote), which motivates heuristic and metaheuristic methods that find good timetables within limited computation.

The second International Timetabling Competition (ITC-2007) included a curriculum-based course timetabling track in which lectures are assigned to periods and rooms, with conflicts determined by instructors and curricula[<sup>[4]</sup>](#mullernote). One solver applied to all three ITC-2007 tracks first constructs a complete timetable and then improves it with hill climbing, great deluge, and optionally simulated annealing<a name="muller"></a>[<sup>[4]</sup>](#mullernote). We adopt a simplified variant: ITC-2007 treats room capacity, minimum working days, and curriculum compactness as soft penalties[<sup>[4]</sup>](#mullernote), whereas we treat room capacity and spreading a course's meetings over different days as hard constraints and measure compactness as idle periods. Our scores will therefore not be presented as official ITC benchmark scores.

Local search represents a candidate timetable as a complete assignment and improves it through small changes<a name="aima"></a>[<sup>[5]</sup>](#aimanote). The min-conflicts heuristic showed that repairing a complete but flawed assignment can solve large constraint satisfaction and scheduling problems, and it was used to schedule Hubble Space Telescope observations<a name="minton"></a>[<sup>[6]</sup>](#mintonnote). Unlike pure hill climbing, simulated annealing sometimes accepts worsening moves, with a probability that falls as a temperature parameter decreases<a name="kirkpatrick"></a>[<sup>[7]</sup>](#kirkpatricknote), and was applied to school timetabling early on<a name="abramson"></a>[<sup>[8]</sup>](#abramsonnote). Genetic algorithms instead search with a population of candidates using selection, crossover, and mutation<a name="holland"></a>[<sup>[9]</sup>](#hollandnote)[<sup>[5]</sup>](#aimanote). A survey of metaheuristics for university timetabling groups methods by how they treat hard versus soft constraints: one-stage methods that optimize a single weighted objective, two-stage methods that reach feasibility before optimizing preferences, and methods that never violate hard constraints but relax another aspect of the problem, such as leaving some events unplaced<a name="lewis"></a>[<sup>[10]</sup>](#lewisnote). Our objective (defined below) follows the weighted approach while weighting feasibility so heavily that it always takes priority.

Rather than comparing methods on a few fixed benchmark instances, we vary resource contention systematically under equal time budgets, which lets us ask how the trade-off between improving one candidate (simulated annealing) and maintaining a population (genetic algorithm) changes as problems become tighter. Because both methods start from the greedy timetable and keep the best candidate found, neither can end worse than greedy; the question is how much each improves and how that changes with contention. We hypothesize that (1) simulated annealing's advantage over hill climbing grows as contention increases, because we expect tighter instances to have more local optima, and (2) the genetic algorithm's population overhead makes it less competitive than simulated annealing under short time limits.

# Problem Statement

**Research question:** Under a fixed computation budget, which method most reliably produces feasible, low-penalty course timetables as resource contention increases?

An instance contains courses, required weekly meetings, instructors, student cohorts, rooms, and available periods. Each meeting lasts one period. Each course has one instructor and may be required by several cohorts (a cohort is a group of students who take the same set of courses, such as one year of a major). A schedule assigns every required meeting to exactly one room and period.

**Hard constraints**
- A room cannot host two meetings in the same period.
- An instructor cannot teach two meetings in the same period.
- Courses sharing a cohort cannot meet simultaneously.
- Meetings must respect instructor availability and room capacity.
- Multiple meetings of a course must occur on different days.

**Soft constraints**
- Reduce idle periods between a cohort's first and last meeting of each day.
- Avoid a fixed set of undesirable periods (the same for all instructors and cohorts; we do not model individual preferences).
- Keep each course in as few different rooms as possible.

Let $H = H_{\text{room}} + H_{\text{inst}} + H_{\text{cohort}} + H_{\text{avail}} + H_{\text{cap}} + H_{\text{day}}$ be the hard-constraint violation count. The first three terms count pairs of meetings in the same period that share a room, an instructor, or at least one cohort (a pair counts once per term, however many cohorts it shares); $H_{\text{avail}}$ and $H_{\text{cap}}$ count meetings placed when their instructor is unavailable or in a room smaller than their enrollment; $H_{\text{day}}$ counts pairs of meetings of the same course on the same day. Define the preference penalty $S = G + U + C$, where
- $G = \sum_{k}\sum_{d} \left(\ell_{kd} - f_{kd} + 1 - n_{kd}\right)$ counts idle periods, with $f_{kd}$, $\ell_{kd}$, and $n_{kd}$ the first period, last period, and number of distinct occupied periods of cohort $k$ on day $d$ (days without meetings contribute 0);
- $U$ counts meetings placed in undesirable periods;
- $C = \sum_{c} \left(|\mathrm{rooms}(c)| - 1\right)$ counts the extra rooms used by each course $c$.

We will document exact counting conventions and keep these unit weights fixed in the primary comparison.

Feasibility takes priority over preferences. Search will minimize $E = M H + S$ with $M = \bar{S} + 1$, where $\bar{S} = 20K + L + (L - N)$ bounds $S$ for $K$ cohorts, $L$ meetings, and $N$ courses (with six periods per day, at most 4 idle periods per cohort-day over 5 days, at most $L$ undesirable meetings, and at most $L - N$ extra rooms). Thus, reducing one hard violation always outweighs any possible preference improvement. Final results will distinguish between feasible and infeasible outputs.

The problem is non-trivial because assignments interact across several shared resources, and even simplified timetabling is NP-complete[<sup>[3]</sup>](#evennote). For $L$ meetings, $P$ periods, and $R$ rooms, the unconstrained assignment space contains $(PR)^L$ possibilities. Our 100-course instances have 200–300 meetings, 30 periods, and 9–25 rooms, so even the smallest has more than $270^{200} > 10^{480}$ possible assignments, and enumeration is impossible.

# Data

Our primary inputs will be **synthetic scheduling instances**, not private university records. An observation is an entire scheduling instance, with linked course, meeting, room, and availability tables.

We plan to generate 1,200 instances: 200 for development and parameter selection and 1,000 held out for evaluation. Proposed instance sizes are 20, 40, 60, and 100 courses, with five teaching days of six periods each (P = 30). Each course will require two or three weekly meetings. We define **resource contention** as room utilization $u = L / (RP)$, the fraction of all room-periods that must be filled. For each course count, we choose the number of rooms $R$ and the mix of two- and three-meeting courses to hit three target levels, low ($u \approx 0.4$), medium ($u \approx 0.6$), and high ($u \approx 0.8$), and we report the realized $u$. Instructor availability, cohort overlap, and enrollments will be drawn from the same distributions at every utilization level, so $u$ is the manipulated factor; we will record realized difficulty descriptors (conflict-graph density, mean instructor availability, meetings per cohort, and number of rooms that fit each course). Development and test instances will be split approximately evenly across the 4 × 3 size–contention cells (about 17 development and 83 test instances per cell).

Recorded fields will include instance ID, generation seed, course ID, instructor ID, cohort IDs, enrollment, required meeting count, meeting ID, room ID, room capacity, day, period, availability, and undesirable-period indicators. Lists and availability masks will be encoded explicitly in JSON; generated configurations and seeds will be versioned.

To guarantee feasibility, we will first construct a valid hidden timetable and derive compatible availability, room capacities, and conflicts from it, adding extra available periods for every instructor so that availability does not reveal the hidden timetable. The hidden timetable is stored separately and never passed to solvers; it serves only as a feasibility certificate, not as an optimum or initialization. Development and test instances use separate seeds and share no hidden timetable or base instance.

We will validate identifiers, references, required meetings, capacities, and the hidden timetable with an independent constraint checker. We will report distributions of instance size and resource contention. Because this is an optimization project rather than supervised prediction, the instances are search problems, not labeled training examples; each is summarized by eight instance-level variables (number of courses $N$, meetings $L$, rooms $R$, utilization $u$, number of cohorts $K$, cohort overlap, instructor availability, and share of undesirable periods). We request approval for this formulation and its synthetic evaluation design; synthetic realism remains a limitation.

# Proposed Solution

We will implement a shared Python representation, objective function, and independent schedule validator, then compare:

1. **Greedy construction baseline.** Repeatedly place the most constrained unplaced meeting, recomputing after each placement how many placements add no hard violation: fewest such placements first, ties broken by most conflicts and then by meeting ID (dynamic minimum-remaining-values ordering from constraint satisfaction[<sup>[5]</sup>](#aimanote)). Choose a feasible placement with the smallest added preference penalty. If no feasible placement exists, use the placement with the smallest added E and record the resulting violations. Use fixed tie-breaking.
2. **Simulated annealing.** Begin from the greedy timetable. Propose moving one meeting or swapping two meetings. Accept improvements and accept a worsening change of size $\Delta E$ with probability $\exp(-\Delta E / T)$[<sup>[7]</sup>](#kirkpatricknote). The initial temperature is set per instance from the $\Delta E$ of sampled worsening moves to reach a target initial acceptance rate, so settings transfer across instance sizes; $T$ then decreases geometrically with the elapsed fraction of the time budget, so instances of different sizes pass through comparable annealing phases, and the best candidate encountered is retained. Because the large weight $M$ can make increases in $H$ effectively unacceptable, we will report acceptance rates separately for moves that change $H$ and moves that change only $S$. As an ablation, we will also run hill climbing with sideways moves: the same initial timetable, moves, time budget, and incremental scoring, but a move is accepted only if $\Delta E \le 0$. This isolates the value of accepting worse moves.
3. **Genetic algorithm with bounded conflict repair.** Encode one room-period assignment per meeting. Initialize a population containing the greedy timetable and randomized candidates. Use tournament selection, course-level crossover (all meetings of a course are copied from the same parent), meeting reassignment mutation, and elitism. After crossover and mutation, repair repeatedly picks a random meeting involved in a hard violation and moves it to the placement with the smallest $E$, for a fixed number of steps per offspring; offspring still infeasible after repair stay in the population with their penalized $E$. Repair time counts toward the runtime budget, and conclusions about the genetic algorithm refer to this combination.

Because H and S are sums of local constraint counts, moving or swapping meetings changes only a few terms. We will compute ΔE incrementally rather than rescoring the full timetable, so every method can evaluate many candidate changes within the time budget. The independent validator will recompute the full score of every reported timetable.

Our main implementation will use Python, NumPy, pandas, and Matplotlib, with pinned dependencies and recorded seeds. The syllabus describes implementation projects as non-trivial tasks solved with multiple AI methods and lists scheduling and optimization tasks as examples; we compare search methods rather than train a predictive model. Simulated annealing and hill climbing were covered in lecture. Genetic algorithms appear in the assigned textbook reading (Russell and Norvig, Chapter 4) but are listed in the syllabus among methods the lectures will not cover, so **we request approval to use a genetic algorithm**.

**Parameter selection:** On the development instances, compare a 3 × 3 grid per method under the same time budget: target initial acceptance rate × final-to-initial temperature ratio for simulated annealing, and population size × mutation strength (expected number of meetings reassigned per offspring) for the genetic algorithm. We will select one global configuration per method: highest mean feasibility rate, with ties broken by mean $S/L$ on development instances where every candidate configuration is feasible. Before evaluating held-out instances, we will freeze the generator settings, constraint and scoring definitions, solver configurations, time budgets, and analysis rules. The score weights will not be tuned to favor a method.

**Scope:** The core deliverable is the greedy, simulated annealing, and genetic algorithm comparison plus the hill-climbing ablation on our documented formulation; if the genetic algorithm is not working by Nov 29, it will be reported as a partial result. An optional extension is to parse a small selection of public timetabling instances after matching their constraints; it is not required for the core project. At proposal time, this repository contains the plan and report template; no working scheduler or experimental results are claimed.

# Evaluation Metrics

All methods will use the same instances, validator, and score. We will report:

- **Feasibility rate:** fraction of runs with $H = 0$.
- **Preference penalty:** $S$ and $S/L$ for feasible outputs, plus the individual $G$, $U$, and $C$ components. Failed runs remain visible and are not treated as low-penalty successes.
- **Computation:** the feasibility curve $F_a(t)$, the fraction of method $a$'s runs that have found a feasible timetable by time $t$; objective evaluations; and best feasible penalty at fixed time budgets. Runs that never become feasible count as failures at the cutoff, not as time samples.
- **Scalability and variability:** results for each course count and contention level u, with median and interquartile range across instances and seeds.

The initial evaluation will allow one second of single-threaded CPU time per run, including initialization and repair but excluding data loading and final validation, on one documented machine, with runs in randomized order. Development-only timing experiments will confirm that every method finishes initialization and performs a meaningful amount of search before budgets are frozen. Each stochastic method will use three seeds per test instance; the deterministic greedy baseline will run once per instance. A preselected stratified subset will also receive separate five-second runs with the same parameter rules to examine improvement with additional time.

We will compare methods on matched instances and show feasibility curves, penalty distributions, and example timetable grids. The instance is the unit of analysis: seeds are summarized per instance, an instance counts as solved by a stochastic method if at least two of its three seeds are feasible, and confidence intervals use bootstrap resampling over instances. Each method's full feasibility rate is the primary result. Quality comparisons are pairwise and conditional: for each pair of methods, we use instances where both are feasible in every run, report how many instances qualify, and compare per-instance median $S/L$. The primary pair is simulated annealing versus the genetic algorithm; the ablation compares simulated annealing with hill climbing. We will test paired differences with Wilcoxon signed-rank tests (Pratt handling of zero differences) and solved-instance rates with McNemar's test, using Holm correction across pairs and cells and reporting median differences with confidence intervals alongside p-values; cells with fewer than 10 qualifying instances will be reported descriptively. For hypothesis (1), we will test for an increasing trend in the gap between simulated annealing and hill climbing across the three contention levels with a Jonckheere–Terpstra test. We will avoid claiming optimality without a proven bound.

# Ethics & Privacy

We will use synthetic instructors and cohorts and collect no personal schedules or student records. Preference weights can favor some groups, and an aggregate score may conceal one cohort receiving a particularly inconvenient timetable. We will therefore also report, for each method, the mean daily idle periods and undesirable-period meetings per cohort and the worst cohort's values, alongside cohort meeting loads. Our simplified constraints omit factors such as accessibility, travel time, and individual accommodations; the resulting timetables will be research demonstrations, not recommendations for actual course allocation.

# Team Expectations

Our working agreement:
- Meet weekly, record decisions in the repository, and flag blockers early.
- Share implementation, experiments, and writing; each member reviews another member's work.
- Agree on the data schema, constraint definitions, and evaluation protocol before implementing separate methods.
- Use development evidence to resolve technical disagreements and document changes to scope.
- Set an internal report deadline at least two days before the official deadline.
- Communicate in our group chat and respond within 24 hours on weekdays.
- If a member cannot finish a task on time, they tell the group at least two days before that task's target date so work can be redistributed; persistent problems will be raised with the TA or instructor.

**Responsibilities:** Derrick Chen and Zhen Wang will build the instance generator, validator, and greedy baseline; Zhen Wang will lead simulated annealing and the hill-climbing ablation; Daira Merino will lead the genetic algorithm; Derrick Chen will lead the experiment runner, statistical analysis, and figures. All three will integrate the code, run experiments, interpret results, and write the report.

# Project Timeline Proposal

Dates follow the course schedule: proposal due Nov 4, exam Nov 20, Thanksgiving Nov 27, and the final report due in finals week (Dec 7–11). We will meet weekly and update this plan at each meeting.

| Target date | Planned work | Lead |
|---|---|---|
| Wed Nov 4 | Submit proposal | All |
| Fri Nov 13 | Finalize constraints and data schema; implement instance generator, validator, and greedy baseline; generate sample instances | Derrick and Zhen; all review |
| Sun Nov 22 | Implement simulated annealing, the hill-climbing ablation, and the genetic algorithm; check that incremental and full scoring agree; verify outputs on small cases (exam on Nov 20) | Zhen (annealing, hill climbing); Daira (genetic algorithm) |
| Sun Nov 29 | Tune on development instances; freeze parameters and evaluation protocol | All |
| Fri Dec 4 | Run held-out comparisons; save raw results; analyze failures, runtime, and schedule quality; draft figures | Derrick; all |
| Two days before the final report deadline | Complete report, figures, and reproduction instructions; review and export final PDF | All |

If the genetic algorithm with repair is not working by Nov 29, we will report it as a partial result and keep the greedy, simulated annealing, and hill-climbing comparison as the core result.

The final report will include background and citations, methods, results, discussion, and conclusion, with a link to this public repository containing source code, generation scripts, configurations, and experiment outputs. Submission will follow the official Gradescope deadline.

# Footnotes

<a name="schaerfnote"></a>1.[^](#schaerf): Schaerf, A. (1999). A survey of automated timetabling. *Artificial Intelligence Review*, 13(2), 87–127. https://doi.org/10.1023/A:1006576209967<br>
<a name="burkenote"></a>2.[^](#burke): Burke, E. K., and Petrovic, S. (2002). Recent research directions in automated timetabling. *European Journal of Operational Research*, 140(2), 266–280. https://doi.org/10.1016/S0377-2217(02)00069-3<br>
<a name="evennote"></a>3.[^](#even): Even, S., Itai, A., and Shamir, A. (1976). On the complexity of timetable and multicommodity flow problems. *SIAM Journal on Computing*, 5(4), 691–703. https://doi.org/10.1137/0205048<br>
<a name="mullernote"></a>4.[^](#muller): Müller, T. (2009). ITC2007 solver description: A hybrid approach. *Annals of Operations Research*, 172(1), 429–446. https://doi.org/10.1007/s10479-009-0644-y ([author's version](https://muller.unitime.org/itc2007.pdf))<br>
<a name="aimanote"></a>5.[^](#aima): Russell, S., and Norvig, P. (2021). *Artificial Intelligence: A Modern Approach* (4th ed.). Pearson. Chapter 4 (local search, simulated annealing, genetic algorithms) and Chapter 6 (constraint satisfaction, min-conflicts). https://aima.cs.berkeley.edu/<br>
<a name="mintonnote"></a>6.[^](#minton): Minton, S., Johnston, M. D., Philips, A. B., and Laird, P. (1992). Minimizing conflicts: A heuristic repair method for constraint satisfaction and scheduling problems. *Artificial Intelligence*, 58(1–3), 161–205. https://doi.org/10.1016/0004-3702(92)90007-K<br>
<a name="kirkpatricknote"></a>7.[^](#kirkpatrick): Kirkpatrick, S., Gelatt, C. D., Jr., and Vecchi, M. P. (1983). Optimization by simulated annealing. *Science*, 220(4598), 671–680. https://doi.org/10.1126/science.220.4598.671<br>
<a name="abramsonnote"></a>8.[^](#abramson): Abramson, D. (1991). Constructing school timetables using simulated annealing: Sequential and parallel algorithms. *Management Science*, 37(1), 98–113. https://doi.org/10.1287/mnsc.37.1.98<br>
<a name="hollandnote"></a>9.[^](#holland): Holland, J. H. (1992). *Adaptation in Natural and Artificial Systems: An Introductory Analysis with Applications to Biology, Control, and Artificial Intelligence*. MIT Press. (Original work published 1975) https://doi.org/10.7551/mitpress/1090.001.0001<br>
<a name="lewisnote"></a>10.[^](#lewis): Lewis, R. (2008). A survey of metaheuristic-based techniques for university timetabling problems. *OR Spectrum*, 30(1), 167–190. https://doi.org/10.1007/s00291-007-0097-0